# Partie 3 : Modélisation et comparaison
Notebook autonome : il recharge le jeu nettoyé (`../data/students_dropout.csv`, sauvegardé en partie 2) ou, à défaut, le télécharge et refait les corrections.

Contenu :
1. Chargement et préparation (mêmes choix qu'en partie 2)
2. Entraînement de 5 modèles avec optimisation des hyperparamètres
3. Tableau comparatif et courbes ROC
4. Effet du seuil de décision
5. Expériences : avec / sans notes S2, détection précoce, variables corrélées
6. Sauvegarde des modèles

## 1. Chargement et préparation

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import joblib

from scipy.stats import loguniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, roc_curve, precision_recall_curve,
                             confusion_matrix)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

In [2]:
CSV = "../data/students_dropout.csv"

if os.path.exists(CSV):
    df = pd.read_csv(CSV)
    print("Données chargées depuis", CSV)
else:
    from ucimlrepo import fetch_ucirepo
    data = fetch_ucirepo(id=697)
    df = data.data.features.copy()
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"Nacionality": "Nationality"})
    df = df.drop(columns=["Educational special needs"])
    df["Dropout"] = (data.data.targets.iloc[:, 0] == "Dropout").astype(int)
    print("Données téléchargées depuis l'UCI")

y = df["Dropout"]
X = df.drop(columns="Dropout")
print(X.shape, "| taux de décrochage :", y.mean().round(3))

Données chargées depuis ../data/students_dropout.csv
(4424, 35) | taux de décrochage : 0.321


In [3]:
CAT_COLS = ["Marital Status", "Application mode", "Course", "Previous qualification",
            "Nationality", "Mother's qualification", "Father's qualification",
            "Mother's occupation", "Father's occupation"]
BIN_COLS = [c for c in X.columns if X[c].nunique() == 2 and c not in CAT_COLS]
NUM_COLS = [c for c in X.columns if c not in CAT_COLS + BIN_COLS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train :", X_train.shape, "| Test :", X_test.shape)


def make_prep(columns):
    """Préprocesseur construit uniquement sur les colonnes données."""
    cat = [c for c in CAT_COLS if c in columns]
    num = [c for c in NUM_COLS if c in columns]
    bin_ = [c for c in BIN_COLS if c in columns]
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False), cat),
        ("num", StandardScaler(), num),
        ("bin", "passthrough", bin_),
    ])

Train : (3539, 35) | Test : (885, 35)


## 2. Les 5 modèles et leurs hyperparamètres
Chaque modèle est dans un `Pipeline` (préparation + modèle). La recherche d'hyperparamètres (`RandomizedSearchCV`, validation croisée stratifiée à 5 plis, critère AUC-ROC) n'utilise donc que le train : **aucune fuite vers le test**.

Le déséquilibre (68 % / 32 %) est géré avec `class_weight="balanced"` et `scale_pos_weight` pour XGBoost.

In [4]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
SPW = neg / pos   # poids de la classe positive pour XGBoost

MODELS = {
    "Régression logistique": (
        LogisticRegression(max_iter=3000, class_weight="balanced"),
        {"clf__C": loguniform(1e-3, 1e1), "clf__penalty": ["l2"]},
    ),
    "Arbre de décision": (
        DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        {"clf__max_depth": [3, 4, 5, 6, 8, 10],
         "clf__min_samples_leaf": [1, 5, 10, 20, 40]},
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"clf__n_neighbors": list(range(3, 40, 2)),
         "clf__weights": ["uniform", "distance"],
         "clf__p": [1, 2]},
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        {"clf__n_estimators": [100, 200, 300],
         "clf__max_depth": [5, 8, 12, None],
         "clf__min_samples_leaf": [1, 3, 5, 10]},
    ),
    "XGBoost": (
        XGBClassifier(scale_pos_weight=SPW, eval_metric="logloss",
                      random_state=RANDOM_STATE, n_jobs=-1),
        {"clf__n_estimators": [100, 200, 300],
         "clf__max_depth": [2, 3, 4, 5],
         "clf__learning_rate": [0.03, 0.05, 0.1],
         "clf__subsample": [0.7, 0.85, 1.0],
         "clf__colsample_bytree": [0.6, 0.8, 1.0]},
    ),
}
print("Modèles :", list(MODELS))
print("scale_pos_weight XGBoost :", round(SPW, 2))

Modèles : ['Régression logistique', 'Arbre de décision', 'KNN', 'Random Forest', 'XGBoost']
scale_pos_weight XGBoost : 2.11


In [5]:
N_ITER = 15   # nombre de combinaisons testées par modèle (augmente pour plus de précision)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fitted = {}      # meilleur pipeline de chaque modèle
best_params = {}
cv_scores = {}
fit_times = {}

for name, (clf, grid) in MODELS.items():
    t0 = time.time()
    pipe = Pipeline([("prep", make_prep(X_train.columns)), ("clf", clf)])
    search = RandomizedSearchCV(pipe, grid, n_iter=N_ITER, scoring="roc_auc",
                                cv=cv, random_state=RANDOM_STATE, n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search.best_estimator_
    best_params[name] = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
    cv_scores[name] = search.best_score_
    fit_times[name] = time.time() - t0
    print(f"{name:22s} AUC (CV) = {search.best_score_:.3f}   ({fit_times[name]:.0f}s)")

Régression logistique  AUC (CV) = 0.922   (5s)
Arbre de décision      AUC (CV) = 0.879   (1s)
KNN                    AUC (CV) = 0.890   (1s)
Random Forest          AUC (CV) = 0.916   (4s)
XGBoost                AUC (CV) = 0.920   (2s)


In [6]:
pd.DataFrame({"Meilleurs hyperparamètres": best_params})

,Meilleurs hyperparamètres
Régression logistique,"{'C': 0.24810409748678125, 'penalty': 'l2'}"
Arbre de décision,"{'min_samples_leaf': 40, 'max_depth': 8}"
KNN,"{'weights': 'distance', 'p': 1, 'n_neighbors':..."
Random Forest,"{'n_estimators': 200, 'min_samples_leaf': 1, '..."
XGBoost,"{'subsample': 0.7, 'n_estimators': 300, 'max_d..."


## 3. Tableau comparatif (évalué sur le jeu de test, seuil 0,5)

In [7]:
def metrics_at(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {
        "AUC-ROC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
        "Précision": precision_score(y_true, pred, zero_division=0),
        "Rappel": recall_score(y_true, pred),
        "F1": f1_score(y_true, pred),
    }

probas = {n: m.predict_proba(X_test)[:, 1] for n, m in fitted.items()}

table = pd.DataFrame({n: metrics_at(y_test, p) for n, p in probas.items()}).T
table.insert(0, "AUC (CV train)", pd.Series(cv_scores))
table["Temps (s)"] = pd.Series(fit_times)
table = table.sort_values("AUC-ROC", ascending=False).round(3)
table

,AUC (CV train),AUC-ROC,PR-AUC,Précision,Rappel,F1,Temps (s)
XGBoost,0.920,0.935,0.905,0.798,0.835,0.816,2.422
Régression logistique,0.922,0.932,0.903,0.792,0.845,0.818,4.533
Random Forest,0.916,0.928,0.897,0.825,0.796,0.810,4.339
KNN,0.890,0.903,0.859,0.952,0.486,0.643,0.748
Arbre de décision,0.879,0.903,0.855,0.749,0.799,0.773,0.693


In [8]:
fig = go.Figure()
for name, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                             name=f"{name} (AUC={roc_auc_score(y_test, p):.3f})"))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines",
                         line=dict(dash="dash", color="gray"), name="Hasard"))
fig.update_layout(title="Courbes ROC des 5 modèles (jeu de test)",
                  xaxis_title="Taux de faux positifs", yaxis_title="Taux de vrais positifs",
                  width=800, height=550)
fig.show()

In [9]:
fig = go.Figure()
for name, p in probas.items():
    prec, rec, _ = precision_recall_curve(y_test, p)
    fig.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name=name))
fig.add_hline(y=y_test.mean(), line_dash="dash", line_color="gray",
              annotation_text="Hasard (taux de positifs)")
fig.update_layout(title="Courbes précision-rappel (jeu de test)",
                  xaxis_title="Rappel", yaxis_title="Précision", width=800, height=550)
fig.show()

In [10]:
best_name = table.index[0]
print("Meilleur modèle (AUC-ROC test) :", best_name)

cm = confusion_matrix(y_test, (probas[best_name] >= 0.5).astype(int))
px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
          x=["Prédit : reste", "Prédit : décroche"],
          y=["Réel : reste", "Réel : décroche"],
          title=f"Matrice de confusion : {best_name} (seuil 0,5)")

Meilleur modèle (AUC-ROC test) : XGBoost


## 4. Effet du seuil de décision
Le seuil 0,5 n'est pas sacré. Le baisser attrape plus de décrocheurs (rappel ↑) au prix de plus de fausses alertes (précision ↓). C'est ce que le **curseur de seuil** de l'application permettra de régler.

In [11]:
rows = []
for t in np.arange(0.1, 0.91, 0.1):
    m = metrics_at(y_test, probas[best_name], t)
    pred = probas[best_name] >= t
    rows.append({"Seuil": round(t, 1), "Étudiants signalés": int(pred.sum()),
                 "Précision": m["Précision"], "Rappel": m["Rappel"], "F1": m["F1"]})
seuils = pd.DataFrame(rows).round(3)
display(seuils)

px.line(seuils, x="Seuil", y=["Précision", "Rappel", "F1"], markers=True,
        title=f"Précision, rappel et F1 selon le seuil ({best_name})")

,Seuil,Étudiants signalés,Précision,Rappel,F1
0,0.1,631,0.441,0.979,0.608
1,0.2,464,0.573,0.937,0.711
2,0.3,371,0.693,0.905,0.785
3,0.4,330,0.752,0.873,0.808
4,0.5,297,0.798,0.835,0.816
5,0.6,269,0.829,0.785,0.807
6,0.7,231,0.887,0.722,0.796
7,0.8,209,0.914,0.673,0.775
8,0.9,161,0.969,0.549,0.701


## 5. Expériences : quelles variables utiliser ?
On compare des **scénarios de variables** avec deux modèles représentatifs (régression logistique et XGBoost, hyperparamètres du meilleur réglage trouvé ci-dessus). Le but est de répondre à trois questions du projet :
- Que perd-on **sans les notes du 2e semestre** ?
- Que reste-t-il pour une **détection précoce** (avant tout résultat) ?
- Supprimer les **variables corrélées** change-t-il quelque chose ?

In [12]:
S1 = [c for c in X.columns if "1st sem" in c]
S2 = [c for c in X.columns if "2nd sem" in c]
CORRELEES = ["Curricular units 2nd sem (credited)", "Curricular units 2nd sem (enrolled)",
             "Curricular units 2nd sem (approved)", "Father's occupation"]

SCENARIOS = {
    "Toutes les variables": [],
    "Sans notes S2": S2,
    "Sans S1 ni S2 (détection précoce)": S1 + S2,
    "Sans variables corrélées": CORRELEES,
}

def build(name, columns):
    clf = MODELS[name][0]
    from sklearn.base import clone
    clf = clone(clf)
    clf.set_params(**best_params[name])
    return Pipeline([("prep", make_prep(columns)), ("clf", clf)])

rows = []
for scen, drop in SCENARIOS.items():
    cols = [c for c in X.columns if c not in drop]
    for name in ["Régression logistique", "XGBoost"]:
        pipe = build(name, cols).fit(X_train[cols], y_train)
        p = pipe.predict_proba(X_test[cols])[:, 1]
        m = metrics_at(y_test, p)
        rows.append({"Scénario": scen, "Modèle": name, "Nb variables": len(cols), **m})

scen_table = pd.DataFrame(rows).round(3)
scen_table

,Scénario,Modèle,Nb variables,AUC-ROC,PR-AUC,Précision,Rappel,F1
0,Toutes les variables,Régression logistique,35,0.932,0.903,0.792,0.845,0.818
1,Toutes les variables,XGBoost,35,0.935,0.905,0.798,0.835,0.816
2,Sans notes S2,Régression logistique,29,0.913,0.873,0.762,0.813,0.787
3,Sans notes S2,XGBoost,29,0.914,0.877,0.753,0.796,0.774
4,Sans S1 ni S2 (détection précoce),Régression logistique,23,0.840,0.752,0.606,0.732,0.663
5,Sans S1 ni S2 (détection précoce),XGBoost,23,0.842,0.764,0.641,0.729,0.682
6,Sans variables corrélées,Régression logistique,31,0.924,0.889,0.790,0.820,0.805
7,Sans variables corrélées,XGBoost,31,0.929,0.895,0.785,0.799,0.792


In [13]:
px.bar(scen_table, x="Scénario", y="AUC-ROC", color="Modèle", barmode="group",
       range_y=[0.5, 1], title="AUC-ROC selon les variables utilisées")

## 6. Sauvegarde des modèles
Ces fichiers serviront à l'API FastAPI et à l'application Streamlit.

In [14]:
os.makedirs("../models", exist_ok=True)
for name, m in fitted.items():
    fname = name.lower().replace(" ", "_").replace("é", "e").replace("è", "e")
    joblib.dump(m, f"../models/{fname}.joblib")
joblib.dump({"cat_cols": CAT_COLS, "bin_cols": BIN_COLS, "num_cols": NUM_COLS,
             "columns": list(X.columns), "best_model": best_name}, "../models/meta.joblib")
table.to_csv("../models/comparatif_modeles.csv")
print("Modèles sauvegardés :", os.listdir("../models"))

Modèles sauvegardés : ['arbre_de_decision.joblib', 'comparatif_modeles.csv', 'knn.joblib', 'meta.joblib', 'random_forest.joblib', 'regression_logistique.joblib', 'xgboost.joblib']


## 7. Bilan de la modélisation

- **Meilleur modèle** : XGBoost (AUC-ROC = 0,935 ; PR-AUC = 0,905 sur le test).
- **Écart entre la régression logistique et XGBoost** : 0,3 point d'AUC (0,932 contre 0,935), et F1 quasi identique (0,818 contre 0,816). Cet écart est dans la marge d'erreur d'un test de 885 étudiants (environ ±1 point) : les deux modèles sont à égalité.
- **Modèle le plus faible** : KNN (AUC 0,903 mais rappel de seulement 0,486 au seuil 0,5, F1 = 0,643). Il est très précis (95 %) mais rate un décrocheur sur deux. Il supporte mal les 119 colonnes issues du one-hot, qui faussent les distances. L'arbre de décision a la même AUC (0,903) mais un F1 plus élevé (0,773), car il est plus équilibré.
- **Effet du retrait des notes S2** : l'AUC passe de 0,935 à 0,914 (XGBoost), soit seulement -2 points.
- **Détection précoce (sans S1 ni S2)** : AUC = 0,842 (XGBoost) et 0,840 (régression logistique), soit -9 points par rapport au modèle complet. Les notes sont l'information la plus prédictive, mais elles arrivent tard.
- **Variables corrélées** : le retrait ne change presque rien (0,929 contre 0,935 pour XGBoost ; 0,924 contre 0,932 pour la régression logistique). On les garde par défaut.
- **Seuil retenu pour l'application** : entre 0,3 et 0,4 (rappel de 87 à 90 %, précision de 69 à 75 %). Rater un décrocheur coûte plus cher qu'une fausse alerte, qui se résume à un appel d'accompagnement.
- **Recommandation provisoire** : la régression logistique perd moins de 0,3 point d'AUC par rapport à XGBoost, pour un modèle bien plus lisible.
- **Limite** : les étudiants « Enrolled » sont classés « pas de décrochage » alors que leur avenir est inconnu.